In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_percentage_error

from loaders._load_vn30_reg_deep import preprocess, VN30, TARGETS
from models.regression.mlp import MultiLayerPerception as MLP

In [3]:
train_loader, valid_loader, test_loader, scaler = preprocess('ACB', 'mlp', val=0.2, verbose=True)  

Train shape: torch.Size([972, 120]), torch.Size([972, 4])
Valid shape: torch.Size([243, 120]), torch.Size([243, 4])
Test shape: torch.Size([328, 120]), torch.Size([328, 4])


In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT_DIR = "checkpoints_mlp"
os.makedirs(CKPT_DIR, exist_ok=True)

def train_one_epoch(model, loader, optimizer, criterion, l1_lambda: float = 0.0, max_grad_norm: float | None = 1.0):
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True).float()
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        if l1_lambda and l1_lambda > 0:
            l1 = sum(p.abs().sum() for p in model.parameters())
            loss = loss + l1_lambda * l1
        loss.backward()
        if max_grad_norm is not None and max_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        bs = yb.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / max(n, 1)

@torch.no_grad()
def evaluate_loss(model, loader, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True).float()
        yb = yb.to(DEVICE, non_blocking=True).float()
        pred = model(xb)
        loss = criterion(pred, yb)
        bs = yb.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / max(n, 1)

@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True).float()
        pred = model(xb).detach().cpu().numpy()
        yb = yb.detach().cpu().numpy()
        preds.append(pred)
        trues.append(yb)
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    return preds, trues

In [5]:
# Random search skip_bits and final training with checkpoints
from typing import Tuple

def _num_skip_bits(num_layers: int) -> int:
    return num_layers * (num_layers - 1) // 2

def _random_bits(n: int, rng: np.random.RandomState) -> list[int]:
    return rng.randint(0, 2, size=n).tolist()

def search(
    symbol: str,
    trials: int = 30,
    epochs_short: int = 5,
    final_epochs: int = 50,
    hidden_dim: int = 16,
    num_layers: int = 4,
    dropout: float = 0.1,
    activation: str = "relu",
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    l1_lambda: float = 0.0,
    use_huber: bool = True,
    huber_delta: float = 1.0,
    max_grad_norm: float | None = 1.0,
    batch_size: int = 32,
    seed: int = 42,
    verbose: bool = True,
) -> Tuple[list[int], float]:
    """
    Random search các skip_bits cho 1 symbol với regularization:
      - Short trials: train epochs_short, track best val; optimizer dùng weight decay, có thể dùng Huber Loss, L1, clip grad.
      - Chọn bits tốt nhất theo best_val.
      - Final: train final_epochs với cùng regularization, lưu checkpoint khi val tốt hơn.

    Returns: (best_bits, best_val_global)
    """
    # Load data
    train_loader, valid_loader, _, _ = preprocess(
        symbol, mode="mlp", batch_size=batch_size, verbose=verbose
    )

    # Infer dims
    xb0, yb0 = next(iter(train_loader))
    input_dim = int(xb0.shape[-1])
    output_dim = int(yb0.shape[-1])

    n_bits = _num_skip_bits(num_layers)
    rng = np.random.RandomState(seed)

    best_bits = None
    best_val_global = float("inf")

    # Criterion factory
    def make_criterion():
        if use_huber:
            return nn.HuberLoss(delta=huber_delta)
        return nn.MSELoss()

    # Short trials
    for t in range(1, trials + 1):
        bits = _random_bits(n_bits, rng)
        model = MLP(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            skip_bits=bits,
            dropout=dropout,
            activation=activation,
        ).to(DEVICE)
        optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = make_criterion()

        best_val_trial = float("inf")
        for ep in range(1, epochs_short + 1):
            _ = train_one_epoch(model, train_loader, optimizer, criterion, l1_lambda=l1_lambda, max_grad_norm=max_grad_norm)
            val_loss = evaluate_loss(model, valid_loader, criterion)
            if val_loss < best_val_trial:
                best_val_trial = val_loss

        if verbose:
            print(f"[{symbol}] trial {t:02d}/{trials} bits={bits} | best_val={best_val_trial:.6f}")

        if best_val_trial < best_val_global:
            best_val_global = best_val_trial
            best_bits = bits

    if trials == -1: best_bits = None

    if verbose:
        print(f"Best bits for {symbol}: {best_bits} | val={best_val_global:.6f}")

    # Final long training with checkpoints
    os.makedirs(CKPT_DIR, exist_ok=True)
    ckpt_path = os.path.join(CKPT_DIR, f"mlp_{symbol}.pth")

    model = MLP(
        input_dim=input_dim,
        output_dim=output_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        skip_bits=best_bits,
        dropout=dropout,
        activation=activation,
    ).to(DEVICE)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = make_criterion()
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    best_val = float("inf")
    best_epoch = -1

    for epoch in range(1, final_epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, l1_lambda=l1_lambda, max_grad_norm=max_grad_norm)
        val_loss = evaluate_loss(model, valid_loader, criterion)
        scheduler.step(val_loss)

        improved = val_loss < best_val - 1e-9
        if improved:
            best_val = val_loss
            best_epoch = epoch
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": {
                        "input_dim": input_dim,
                        "output_dim": output_dim,
                        "hidden_dim": hidden_dim,
                        "num_layers": num_layers,
                        "skip_bits": best_bits,
                        "dropout": dropout,
                        "activation": activation,
                        "lr": lr,
                        "weight_decay": weight_decay,
                        "l1_lambda": l1_lambda,
                        "use_huber": use_huber,
                        "huber_delta": huber_delta,
                        "max_grad_norm": max_grad_norm,
                    },
                    "val_loss": float(best_val),
                    "epoch": int(best_epoch),
                },
                ckpt_path,
            )

        if verbose and (epoch % 10 == 0 or improved):
            print(
                f"Epoch {epoch:03d} | train {tr_loss:.6f} | val {val_loss:.6f}" +
                ("  <-- best" if improved else "")
            )

    if verbose:
        print(
            f"Saved best to {ckpt_path} | best_val={best_val:.6f} @ epoch {best_epoch}"
        )

    return best_bits, best_val_global

In [6]:
def eval(symbol: str):
    _, _, test_loader, scaler = preprocess(symbol, mode="mlp", batch_size=32, verbose=False)

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    CKPT_PATH = f"checkpoints_mlp/mlp_{symbol}.pth"

    # Khởi tạo model cùng cấu hình như khi train, rồi load state
    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    cfg = ckpt["config"]

    model = MLP(
        input_dim=cfg["input_dim"],
        output_dim=cfg["output_dim"],
        hidden_dim=cfg["hidden_dim"],
        num_layers=cfg["num_layers"],
        skip_bits=cfg["skip_bits"],
        dropout=cfg["dropout"],
        activation=cfg["activation"],
    ).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    pred_scaled, true_scaled = predict_all(model, test_loader)  # shape [N, 4]

    y_pred = scaler.inverse_transform(pred_scaled)
    y_true = scaler.inverse_transform(true_scaled)

    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    print(f"Symbol {symbol} | R2: {r2:.4f} | MAPE: {mape:.2f}%")

    return r2, mape

In [7]:
tracks = {"r2": [], "mape": [], "r2_origin": [], "mape_origin": []}

for symbol in VN30:
    _ = search(symbol, trials=-1, verbose=False)
    r2, mape = eval(symbol)
    tracks["r2_origin"].append(r2)
    tracks["mape_origin"].append(mape)
    _ = search(symbol, trials=10, verbose=False)
    r2, mape = eval(symbol)
    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R2 (original): {np.mean(tracks['r2_origin']):.4f} | Mean MAPE (original): {np.mean(tracks['mape_origin']):.2f}%")
print(f"Std R2 (original): {np.std(tracks['r2_origin']):.4f} | Std MAPE (original): {np.std(tracks['mape_origin']):.2f}%")
print(f"Mean R2: {np.mean(tracks['r2']):.4f} | Mean MAPE: {np.mean(tracks['mape']):.2f}%")
print(f"Std R2: {np.std(tracks['r2']):.4f} | Std MAPE: {np.std(tracks['mape']):.2f}%")

Symbol ACB | R2: -1.2338 | MAPE: 7.64%
Symbol ACB | R2: 0.8590 | MAPE: 1.49%
Symbol BCM | R2: 0.8350 | MAPE: 2.78%
Symbol BCM | R2: 0.9114 | MAPE: 1.95%
Symbol BID | R2: 0.7153 | MAPE: 2.09%
Symbol BID | R2: 0.8686 | MAPE: 1.38%
Symbol BVH | R2: 0.9468 | MAPE: 1.85%
Symbol BVH | R2: 0.9471 | MAPE: 1.71%
Symbol CTG | R2: 0.6967 | MAPE: 4.31%
Symbol CTG | R2: 0.9217 | MAPE: 1.99%
Symbol FPT | R2: 0.7319 | MAPE: 6.69%
Symbol FPT | R2: 0.9576 | MAPE: 2.70%
Symbol GAS | R2: 0.8951 | MAPE: 1.26%
Symbol GAS | R2: 0.8593 | MAPE: 1.40%
Symbol GVR | R2: 0.8232 | MAPE: 4.60%
Symbol GVR | R2: 0.8966 | MAPE: 3.46%
Symbol HDB | R2: 0.7163 | MAPE: 4.63%
Symbol HDB | R2: 0.7332 | MAPE: 4.40%
Symbol HPG | R2: 0.4544 | MAPE: 2.82%
Symbol HPG | R2: 0.7892 | MAPE: 1.64%
Symbol LPB | R2: 0.9469 | MAPE: 4.70%
Symbol LPB | R2: 0.9791 | MAPE: 2.98%
Symbol MBB | R2: 0.6115 | MAPE: 4.10%
Symbol MBB | R2: 0.9010 | MAPE: 1.75%
Symbol MSN | R2: 0.8829 | MAPE: 1.80%
Symbol MSN | R2: 0.9121 | MAPE: 1.53%
Symbol MWG 